# EpiCite — Epistemic Sentence Classifier & Citation Need Scorer
**Group 12 | AIT NLP Course**  
Biraj Koirala · Longfei Shi

---
### Pipeline Overview
```
Raw Text  →  Feature Extraction (12 features)
          →  Stage 1: DistilBERT (epistemic type classification)
          →  Stage 2: XGBoost + Isotonic Calibration (citation need score 0–1)
          →  SHAP Explainability
```

### Notebook Structure
| # | Section |
|---|---------|
| 1 | Install & Imports |
| 2 | Dataset Loading & Label Mapping |
| 3 | Feature Extraction (12 linguistic features) |
| 4 | Stage 1 — DistilBERT Fine-tuning |
| 5 | Stage 2 — Citation Need Scorer (XGBoost) |
| 6 | Ablation Study |


## 1 · Install Dependencies

In [1]:
# ── Run once in Colab ─────────────────────────────────────────────────────────
!pip install -q transformers datasets torch scikit-learn xgboost shap spacy nltk
!python -m spacy download en_core_web_sm -q


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# NLP
import spacy
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

# ML
from sklearn.metrics import (classification_report, f1_score,
                              accuracy_score, brier_score_loss,
                              confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
import xgboost as xgb
import shap

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                           DistilBertForSequenceClassification,
                           get_linear_schedule_with_warmup)
from torch.optim import AdamW

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED        = 42
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS_S1   = 4
LR_S1       = 2e-5

torch.manual_seed(SEED)
np.random.seed(SEED)

# Five-class taxonomy
LABEL2ID = {"Claim": 0, "Fact": 1, "Evidence": 2, "Opinion": 3, "Background": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = len(LABEL2ID)

print(f"Device: {DEVICE}")
print(f"Label taxonomy: {list(LABEL2ID.keys())}")


## 2 · Dataset Loading & Label Mapping

We combine two corpora for Stage 1 training:
- **IBM Claim Detection** (~2,500 sentences): Claim, Evidence, Background
- **UKP Argument Mining** (~25,000 sentences): Argument / Non-Argument

Both are remapped to the unified **5-class taxonomy**.

| Source label | → Unified label | Condition |
|---|---|---|
| IBM Claim | → Claim | — |
| IBM Evidence | → Evidence | — |
| IBM Background | → Background | — |
| UKP Argument | → Claim | no citation marker present |
| UKP Argument | → Evidence | citation marker present |
| UKP Non-Argument | → Opinion | contains first-person pronoun |
| UKP Non-Argument | → Background | otherwise |


In [ ]:
# ── Label mapping helpers ─────────────────────────────────────────────────────
CITATION_MARKERS = re.compile(
    r'\b(according to|cited by|as shown|as reported|as stated|'
    r'references|\(\d{4}\)|\[\d+\])\b', re.IGNORECASE)
FIRST_PERSON     = re.compile(r'\b(I|me|my|we|our|us)\b')

def remap_ibm(row):
    mapping = {"claim": "Claim", "evidence": "Evidence", "background": "Background"}
    return mapping.get(str(row["label"]).lower(), "Background")

def remap_ukp(row):
    text = str(row["sentence"])
    if str(row["label"]).lower() in ("argument", "1"):
        return "Evidence" if CITATION_MARKERS.search(text) else "Claim"
    else:
        return "Opinion" if FIRST_PERSON.search(text) else "Background"


In [ ]:
# ── Load IBM Claim Detection ──────────────────────────────────────────────────
# Expected columns: sentence, label  (claim / evidence / background)
# Download from: https://research.ibm.com/haifa/dept/vst/debating_data.shtml
# Place at: data/ibm_claim_detection.csv

IBM_PATH = "data/ibm_claim_detection.csv"

if os.path.exists(IBM_PATH):
    ibm_df = pd.read_csv(IBM_PATH)
    ibm_df = ibm_df.rename(columns={"text": "sentence"})[["sentence", "label"]]
    ibm_df["unified_label"] = ibm_df.apply(remap_ibm, axis=1)
    print(f"IBM loaded:  {len(ibm_df):,} rows")
    print(ibm_df["unified_label"].value_counts())
else:
    # ── Synthetic stub so the notebook runs end-to-end without data ──────────
    print("⚠  IBM file not found — using synthetic stub for structure testing.")
    import random
    random.seed(SEED)
    STUB_LABELS = ["Claim", "Evidence", "Background"]
    ibm_df = pd.DataFrame({
        "sentence": [f"IBM stub sentence number {i}." for i in range(2500)],
        "unified_label": [random.choice(STUB_LABELS) for _ in range(2500)]
    })


In [ ]:
# ── Load UKP Argument Mining ──────────────────────────────────────────────────
# Expected columns: sentence, label  (Argument / NoArgument)
# Download from: https://github.com/UKPLab/acl2014-argument-mining
# Place at: data/ukp_essays.csv

UKP_PATH = "data/ukp_essays.csv"

if os.path.exists(UKP_PATH):
    ukp_df = pd.read_csv(UKP_PATH)
    ukp_df = ukp_df.rename(columns={"text": "sentence"})[["sentence", "label"]]
    ukp_df["unified_label"] = ukp_df.apply(remap_ukp, axis=1)
    print(f"UKP loaded:  {len(ukp_df):,} rows")
    print(ukp_df["unified_label"].value_counts())
else:
    print("⚠  UKP file not found — using synthetic stub.")
    random.seed(SEED + 1)
    UKP_STUB_LABELS = ["Claim", "Opinion", "Background", "Evidence"]
    ukp_df = pd.DataFrame({
        "sentence": [f"UKP stub sentence number {i}." for i in range(25000)],
        "unified_label": [random.choice(UKP_STUB_LABELS) for _ in range(25000)]
    })


In [ ]:
# ── Merge Stage 1 corpus ──────────────────────────────────────────────────────
stage1_df = (
    pd.concat([
        ibm_df[["sentence", "unified_label"]],
        ukp_df[["sentence", "unified_label"]]
    ], ignore_index=True)
    .dropna(subset=["sentence", "unified_label"])
    .drop_duplicates(subset=["sentence"])
    .reset_index(drop=True)
)
stage1_df["label_id"] = stage1_df["unified_label"].map(LABEL2ID)

print(f"\nStage 1 combined: {len(stage1_df):,} sentences")
print("\nClass distribution:")
print(stage1_df["unified_label"].value_counts())

# Train / val / test  80/10/10
s1_train, s1_temp  = train_test_split(stage1_df, test_size=0.20, random_state=SEED,
                                       stratify=stage1_df["label_id"])
s1_val,   s1_test  = train_test_split(s1_temp,   test_size=0.50, random_state=SEED,
                                       stratify=s1_temp["label_id"])
print(f"\nSplits — train: {len(s1_train):,}  val: {len(s1_val):,}  test: {len(s1_test):,}")


In [ ]:
# ── Load WikiSQE for Stage 2 ──────────────────────────────────────────────────
# Paper: Garbacea & Mei, 2023  |  arXiv:2305.01074
# HuggingFace: datasets.load_dataset("garbacea/wikisqe") or CSV download
# Expected cols: sentence, citation_needed (0/1)

WIKISQE_PATH = "data/wikisqe.csv"

if os.path.exists(WIKISQE_PATH):
    wiki_df = pd.read_csv(WIKISQE_PATH)
    wiki_df = wiki_df[["sentence", "citation_needed"]].dropna().reset_index(drop=True)
    print(f"WikiSQE loaded: {len(wiki_df):,} rows")
    print(wiki_df["citation_needed"].value_counts())
else:
    # Try HuggingFace datasets
    try:
        from datasets import load_dataset
        print("Trying HuggingFace datasets...")
        hf_ds = load_dataset("garbacea/wikisqe", split="train")
        wiki_df = hf_ds.to_pandas()[["sentence", "citation_needed"]]
        print(f"WikiSQE loaded from HF: {len(wiki_df):,} rows")
    except Exception:
        print("⚠  WikiSQE not found — using synthetic stub.")
        wiki_df = pd.DataFrame({
            "sentence": [f"Wiki stub sentence {i}." for i in range(150000)],
            "citation_needed": np.random.RandomState(SEED).randint(0, 2, 150000)
        })


## 3 · Feature Extraction — 12 Linguistic Features

| # | Category | Feature | Description |
|---|---|---|---|
| 1 | Lexical | `vague_quantifiers` | Count of hedged quantity words (some, many, few…) |
| 2 | Lexical | `hedging_words` | Modal / epistemic hedge density (might, possibly…) |
| 3 | Lexical | `superlative_adj` | Count of JJS POS tags |
| 4 | Syntactic | `passive_voice` | Binary: contains passive construction |
| 5 | Syntactic | `nominalization_rate` | Ratio of nominalizations (−tion/−ment/−ness…) |
| 6–9 | Syntactic | `pos_noun/verb/adj/adv` | POS ratios per sentence |
| 10 | Contextual | `sentence_position` | Normalized position in document (0–1) |
| 11 | Contextual | `citation_density` | Citation marker count in ±2-sentence window |
| 12 | Contextual | `para_relative_idx` | Normalized position within paragraph (0–1) |


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner"])

VAGUE_QUANTIFIERS = {
    "some", "many", "few", "several", "various", "numerous",
    "certain", "most", "much", "little", "a lot", "lots"
}
HEDGING_WORDS = {
    "might", "may", "could", "possibly", "probably", "perhaps",
    "seems", "appears", "suggests", "reportedly", "allegedly",
    "presumably", "arguably", "tend", "tends", "typically",
    "generally", "often", "sometimes", "usually"
}
NOMINALIZATION_SUFFIXES = ("tion", "ment", "ness", "ity", "ance", "ence",
                            "ism", "age", "al")


def extract_features(sentence: str,
                     doc_sentences: list = None,
                     sent_idx: int = 0) -> dict:
    '''
    Returns a dict of 12 numeric linguistic features for one sentence.

    Parameters
    ----------
    sentence     : the sentence text
    doc_sentences: all sentences in the document (for contextual features)
    sent_idx     : index of this sentence in doc_sentences
    '''
    doc = nlp(sentence)
    tokens = [t for t in doc if not t.is_space]
    n = len(tokens) if tokens else 1

    # ── Lexical ───────────────────────────────────────────────────────────────
    lower_toks = [t.text.lower() for t in tokens]
    vague_q   = sum(1 for t in lower_toks if t in VAGUE_QUANTIFIERS)
    hedge_cnt = sum(1 for t in lower_toks if t in HEDGING_WORDS)
    super_adj = sum(1 for t in doc if t.tag_ == "JJS")

    # ── Syntactic ─────────────────────────────────────────────────────────────
    # Passive: aux + nsubjpass or auxpass dependency
    passive = int(any(t.dep_ in ("nsubjpass", "auxpass") for t in doc))

    # Nominalization rate
    nomin = sum(1 for t in tokens
                if t.text.lower().endswith(NOMINALIZATION_SUFFIXES)
                and t.pos_ == "NOUN")
    nomin_rate = nomin / n

    # POS distribution
    pos_counts = {"NOUN": 0, "VERB": 0, "ADJ": 0, "ADV": 0}
    for t in tokens:
        if t.pos_ in pos_counts:
            pos_counts[t.pos_] += 1
    pos_noun = pos_counts["NOUN"] / n
    pos_verb = pos_counts["VERB"] / n
    pos_adj  = pos_counts["ADJ"]  / n
    pos_adv  = pos_counts["ADV"]  / n

    # ── Contextual ────────────────────────────────────────────────────────────
    if doc_sentences:
        total_sents = len(doc_sentences)
        sent_pos    = sent_idx / max(total_sents - 1, 1)   # [0, 1]

        # Citation density in ±2-sentence window
        window = doc_sentences[max(0, sent_idx-2): sent_idx+3]
        window_text = " ".join(window)
        cit_density = len(CITATION_MARKERS.findall(window_text))
    else:
        sent_pos    = 0.0
        cit_density = 0

    # Paragraph-relative index — approximate by assuming paragraphs
    # are separated by sentences starting with capital after a period.
    # (Real use: pass para_idx / para_len explicitly.)
    para_rel_idx = sent_pos   # fallback; override when calling with para info

    return {
        "vague_quantifiers":  vague_q,
        "hedging_words":      hedge_cnt,
        "superlative_adj":    super_adj,
        "passive_voice":      passive,
        "nominalization_rate": nomin_rate,
        "pos_noun":           pos_noun,
        "pos_verb":           pos_verb,
        "pos_adj":            pos_adj,
        "pos_adv":            pos_adv,
        "sentence_position":  sent_pos,
        "citation_density":   cit_density,
        "para_relative_idx":  para_rel_idx,
    }

FEATURE_COLS = list(extract_features("test sentence").keys())
print("Feature columns:", FEATURE_COLS)
print("Total features:", len(FEATURE_COLS))


In [ ]:
def extract_features_batch(sentences: list,
                            batch_size: int = 512,
                            treat_each_as_doc: bool = True) -> pd.DataFrame:
    # Extracts 12 features for a list of sentences.
    # When treat_each_as_doc=True each sentence is its own document.
    records = []
    for i, sent in enumerate(sentences):
        doc_sents = [sent] if treat_each_as_doc else sentences
        idx       = 0      if treat_each_as_doc else i
        records.append(extract_features(str(sent), doc_sents, idx))
        if (i + 1) % 1000 == 0:
            print(f"  Extracted {i+1:,}/{len(sentences):,}", end="\r")
    print()
    return pd.DataFrame(records)


# ── Extract for Stage 1 training set (for use in Stage 2 fallback & ablation)
print("Extracting features for Stage 1 training set...")
s1_feats_train = extract_features_batch(s1_train["sentence"].tolist())
s1_feats_val   = extract_features_batch(s1_val["sentence"].tolist())
s1_feats_test  = extract_features_batch(s1_test["sentence"].tolist())

print("\nExtracted feature matrix shape:", s1_feats_train.shape)
s1_feats_train.head(3)


## 4 · Stage 1 — DistilBERT Fine-tuning

We fine-tune `distilbert-base-uncased` on the combined IBM+UKP corpus  
(27,500 sentences, 5-class taxonomy).

**Training details:**
- Optimizer: AdamW, lr = 2e-5
- Scheduler: Linear warmup (10 % of steps)
- Batch size: 32  |  Max sequence length: 128
- Metric to optimise: **macro F1** (target > 0.72)


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class EpistemicDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len=MAX_LEN):
        self.sentences  = list(sentences)
        self.labels     = list(labels)
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_ds  = EpistemicDataset(s1_train["sentence"], s1_train["label_id"], tokenizer)
val_ds    = EpistemicDataset(s1_val["sentence"],   s1_val["label_id"],   tokenizer)
test_ds   = EpistemicDataset(s1_test["sentence"],  s1_test["label_id"],  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
model_s1 = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

# ── Optimiser & scheduler ─────────────────────────────────────────────────────
optimizer = AdamW(model_s1.parameters(), lr=LR_S1, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS_S1
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.10 * total_steps),
    num_training_steps=total_steps
)

print(f"Parameters: {sum(p.numel() for p in model_s1.parameters()):,}")
print(f"Total training steps: {total_steps}")


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for batch in loader:
        input_ids  = batch["input_ids"].to(device)
        attn_mask  = batch["attention_mask"].to(device)
        labels     = batch["labels"].to(device)

        optimizer.zero_grad()
        out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = out.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1


def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            total_loss += out.loss.item()
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
CKPT_PATH  = "epicite_stage1_best.pt"
best_val_f1 = 0.0
history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

for epoch in range(1, EPOCHS_S1 + 1):
    tr_loss, tr_f1 = train_epoch(model_s1, train_loader, optimizer, scheduler, DEVICE)
    vl_loss, vl_f1, _, _, _ = evaluate(model_s1, val_loader, DEVICE)

    history["train_loss"].append(tr_loss)
    history["train_f1"].append(tr_f1)
    history["val_loss"].append(vl_loss)
    history["val_f1"].append(vl_f1)

    improved = "✓ saved" if vl_f1 > best_val_f1 else ""
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model_s1.state_dict(), CKPT_PATH)

    print(f"Epoch {epoch}/{EPOCHS_S1}  "
          f"train_loss={tr_loss:.4f}  train_f1={tr_f1:.4f}  "
          f"val_loss={vl_loss:.4f}  val_f1={vl_f1:.4f}  {improved}")

print(f"\n✅ Best val macro-F1: {best_val_f1:.4f}")


In [ ]:
# ── Learning curve ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = range(1, EPOCHS_S1 + 1)

axes[0].plot(epochs_x, history["train_loss"], label="Train")
axes[0].plot(epochs_x, history["val_loss"],   label="Val")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history["train_f1"], label="Train")
axes[1].plot(epochs_x, history["val_f1"],   label="Val")
axes[1].axhline(0.72, color="red", linestyle="--", label="Target F1 = 0.72")
axes[1].set(title="Macro F1", xlabel="Epoch", ylabel="F1")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("stage1_learning_curve.png", dpi=150)
plt.show()


In [ ]:
# ── Test-set evaluation ───────────────────────────────────────────────────────
model_s1.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
_, test_f1, test_preds, test_labels, test_probs = evaluate(model_s1, test_loader, DEVICE)

print("=" * 60)
print(f"Stage 1  TEST  Macro-F1: {test_f1:.4f}")
print("=" * 60)
print(classification_report(test_labels, test_preds,
                             target_names=list(LABEL2ID.keys())))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()), ax=ax)
ax.set(title="Stage 1 Confusion Matrix", xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.savefig("stage1_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# ── Utility: run Stage 1 inference on any DataFrame ──────────────────────────
def predict_stage1(sentences: list, batch_size: int = 64):
    """
    Returns (pred_labels, prob_matrix) for a list of sentences.
    pred_labels : np.array of int  (label IDs)
    prob_matrix : np.array (N, 5)  (softmax probabilities per class)
    """
    model_s1.eval()
    all_preds, all_probs = [], []
    dataset = EpistemicDataset(sentences, [0]*len(sentences), tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in loader:
            out   = model_s1(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE)
            )
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(probs.argmax(axis=-1))

    return np.array(all_preds), np.vstack(all_probs)


print("predict_stage1() ready.")


## 5 · Stage 2 — Citation Need Scorer

**Input vector (18 dimensions):**
- 12 linguistic features (from Section 3)
- 5 epistemic class probabilities from Stage 1
- 1 Stage 1 confidence score (max probability)

**Model:** XGBoost + Isotonic Regression calibration  
**Output:** calibrated probability ∈ [0, 1]

> Note: Per TA feedback, this stage uses a **single, well-motivated model** (XGBoost)
> rather than an experiment table of multiple classifiers.  
> Model selection rationale: gradient boosting handles mixed numerical features well,
> is robust to moderate class imbalance, and produces feature importances that
> feed naturally into SHAP explanations.


In [ ]:
# ── Build Stage 2 feature matrix for WikiSQE ─────────────────────────────────
print("Step 1/3: Extracting linguistic features from WikiSQE...")
wiki_ling_feats = extract_features_batch(wiki_df["sentence"].tolist())

print("Step 2/3: Running Stage 1 inference on WikiSQE...")
wiki_pred_ids, wiki_pred_probs = predict_stage1(wiki_df["sentence"].tolist())
wiki_confidence = wiki_pred_probs.max(axis=1, keepdims=True)

# Epistemic class probabilities (5 cols) + confidence (1 col)
wiki_epi_feats = pd.DataFrame(
    np.hstack([wiki_pred_probs, wiki_confidence]),
    columns=[f"epi_prob_{c}" for c in LABEL2ID.keys()] + ["epi_confidence"]
)

# Full 18-dim input
X_wiki = pd.concat([wiki_ling_feats.reset_index(drop=True),
                    wiki_epi_feats.reset_index(drop=True)], axis=1)
y_wiki = wiki_df["citation_needed"].values

print(f"Step 3/3: Feature matrix ready. Shape: {X_wiki.shape}")
print(f"Citation-needed rate: {y_wiki.mean():.2%}")


In [ ]:
# ── Train / val / test split for Stage 2 ─────────────────────────────────────
X_s2_train, X_s2_temp, y_s2_train, y_s2_temp = train_test_split(
    X_wiki, y_wiki, test_size=0.20, random_state=SEED, stratify=y_wiki
)
X_s2_val, X_s2_test, y_s2_val, y_s2_test = train_test_split(
    X_s2_temp, y_s2_temp, test_size=0.50, random_state=SEED, stratify=y_s2_temp
)

print(f"Stage 2 splits — train: {len(y_s2_train):,}  "
      f"val: {len(y_s2_val):,}  test: {len(y_s2_test):,}")


In [ ]:
# ── XGBoost base classifier ───────────────────────────────────────────────────
xgb_base = xgb.XGBClassifier(
    n_estimators    = 500,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    scale_pos_weight= (y_s2_train == 0).sum() / (y_s2_train == 1).sum(),
    use_label_encoder=False,
    eval_metric     ="logloss",
    random_state    = SEED,
    n_jobs          = -1
)

# ── Isotonic calibration ──────────────────────────────────────────────────────
model_s2 = CalibratedClassifierCV(xgb_base, method="isotonic", cv=3)

print("Training Stage 2 model (XGBoost + Isotonic calibration)...")
model_s2.fit(X_s2_train, y_s2_train)
print("✅ Done.")


In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
y_pred_s2  = model_s2.predict(X_s2_test)
y_prob_s2  = model_s2.predict_proba(X_s2_test)[:, 1]

test_f1_s2 = f1_score(y_s2_test, y_pred_s2)
brier      = brier_score_loss(y_s2_test, y_prob_s2)

print("=" * 60)
print(f"Stage 2  TEST  Binary F1 : {test_f1_s2:.4f}")
print(f"Stage 2  TEST  Brier Score (calibration, ↓): {brier:.4f}")
print("=" * 60)
print(classification_report(y_s2_test, y_pred_s2,
                             target_names=["Not needed", "Citation needed"]))

# ── Calibration plot ──────────────────────────────────────────────────────────
from sklearn.calibration import calibration_curve

fraction_pos, mean_pred = calibration_curve(y_s2_test, y_prob_s2, n_bins=10)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_pred, fraction_pos, "s-", label="XGBoost + Isotonic")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set(title="Calibration Curve — Stage 2",
       xlabel="Mean predicted probability",
       ylabel="Fraction of positives")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("stage2_calibration.png", dpi=150)
plt.show()


In [ ]:
# ── SHAP explainability ───────────────────────────────────────────────────────
# Use the underlying XGBoost estimator for SHAP (calibration wrapper doesn't
# expose a SHAP-compatible interface directly).
xgb_fitted = model_s2.calibrated_classifiers_[0].estimator

explainer    = shap.TreeExplainer(xgb_fitted)
shap_values  = explainer.shap_values(X_s2_test.iloc[:500])   # sample for speed

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_s2_test.iloc[:500],
                  feature_names=X_wiki.columns.tolist(),
                  plot_type="bar", show=False)
plt.title("SHAP Feature Importance — Stage 2")
plt.tight_layout()
plt.savefig("shap_importance.png", dpi=150)
plt.show()

# ── Top features ──────────────────────────────────────────────────────────────
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df   = pd.DataFrame({
    "feature":    X_wiki.columns.tolist(),
    "mean_|SHAP|": mean_shap
}).sort_values("mean_|SHAP|", ascending=False)

print("\nTop 10 features by mean |SHAP|:")
print(shap_df.head(10).to_string(index=False))


## 6 · Ablation Study

This section directly addresses **TA Feedback #3** (missing ablation studies).

We isolate the contribution of each component of the Stage 2 input vector
by training four variants and comparing their test F1 and Brier scores.

| Variant | Features used | Purpose |
|---|---|---|
| **A — Ling only** | 12 linguistic features only | Baseline without epistemic type |
| **B — Epi only** | Epistemic probs + confidence only (6 dims) | Epistemic signal alone |
| **C — Full (predicted)** | All 18 features (Ling + Epi predicted) | Full pipeline |
| **D — Full (oracle)** | All 18 features (Ling + Epi gold labels) | Performance ceiling |

If **H1** holds (epistemic type > surface features), we expect:  
`B > A` and `C > A` in macro F1.


In [ ]:
# ── Oracle (gold) Stage 1 labels for WikiSQE ─────────────────────────────────
# In practice, WikiSQE doesn't ship epistemic labels — we simulate oracle
# labels by using the *test-set* Stage 1 predictions as a proxy.
# For a rigorous oracle, annotate a subset of WikiSQE manually with
# the 5-class taxonomy and use those gold labels here.

# For now: oracle = Stage 1 predictions on train split
# (avoids data leakage since Stage 1 was trained on IBM/UKP, not WikiSQE)
wiki_oracle_probs = wiki_pred_probs.copy()  # replace with gold when available
wiki_oracle_conf  = wiki_confidence.copy()

# ── Build ablation feature matrices ──────────────────────────────────────────
wiki_ling_np = wiki_ling_feats.values
wiki_epi_np  = np.hstack([wiki_pred_probs, wiki_confidence])

ablation_sets = {
    "A_ling_only":       wiki_ling_np,
    "B_epi_only":        wiki_epi_np,
    "C_full_predicted":  np.hstack([wiki_ling_np, wiki_epi_np]),
    "D_full_oracle":     np.hstack([wiki_ling_np,
                                    np.hstack([wiki_oracle_probs, wiki_oracle_conf])]),
}

ablation_results = []

for name, X_full in ablation_sets.items():
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_full, y_wiki, test_size=0.20, random_state=SEED, stratify=y_wiki
    )
    X_vl, X_te, y_vl, y_te = train_test_split(
        X_tmp,  y_tmp,  test_size=0.50, random_state=SEED, stratify=y_tmp
    )

    clf = CalibratedClassifierCV(
        xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(y_tr == 0).sum() / max((y_tr == 1).sum(), 1),
            use_label_encoder=False, eval_metric="logloss",
            random_state=SEED, n_jobs=-1
        ),
        method="isotonic", cv=3
    )
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    f1     = f1_score(y_te, y_pred)
    brier  = brier_score_loss(y_te, y_prob)
    acc    = accuracy_score(y_te, y_pred)

    ablation_results.append({
        "Variant": name,
        "Features": X_full.shape[1],
        "F1":    round(f1,    4),
        "Acc":   round(acc,   4),
        "Brier": round(brier, 4),
    })
    print(f"  {name:25s}  F1={f1:.4f}  Brier={brier:.4f}")

ablation_df = pd.DataFrame(ablation_results)
print("\n" + "=" * 60)
print(ablation_df.to_string(index=False))


In [ ]:
# ── Ablation bar chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
labels = ablation_df["Variant"].str.replace("_", " ").str.title()

axes[0].bar(labels, ablation_df["F1"], color=colors, edgecolor="white", width=0.6)
axes[0].set(title="Ablation — Binary F1 (↑)", ylim=(0, 1),
            xlabel="Variant", ylabel="F1 Score")
for ax_idx, val in enumerate(ablation_df["F1"]):
    axes[0].text(ax_idx, val + 0.01, f"{val:.3f}", ha="center", fontsize=10)

axes[1].bar(labels, ablation_df["Brier"], color=colors, edgecolor="white", width=0.6)
axes[1].set(title="Ablation — Brier Score (↓)",
            xlabel="Variant", ylabel="Brier Score")
for ax_idx, val in enumerate(ablation_df["Brier"]):
    axes[1].text(ax_idx, val + 0.002, f"{val:.3f}", ha="center", fontsize=10)

plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")
plt.setp(axes[1].get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.savefig("ablation_results.png", dpi=150)
plt.show()

print("\nKey takeaway:")
delta = ablation_df.loc[ablation_df["Variant"]=="C_full_predicted", "F1"].values[0] \
      - ablation_df.loc[ablation_df["Variant"]=="A_ling_only",       "F1"].values[0]
print(f"  Full pipeline vs. Ling-only ΔF1 = {delta:+.4f}")
print(f"  This {'supports' if delta > 0 else 'contradicts'} H1 "
      f"(epistemic type adds signal beyond surface features).")


In [ ]:
# ── Per-class contribution analysis (supports RQ2) ───────────────────────────
# Show which epistemic type drives the highest citation-need rate in WikiSQE
epi_label_names = [ID2LABEL[i] for i in range(NUM_CLASSES)]
wiki_pred_labels = wiki_pred_ids  # from predict_stage1() above

citation_by_type = {}
for i, name in enumerate(epi_label_names):
    mask = (wiki_pred_labels == i)
    if mask.sum() > 0:
        citation_by_type[name] = y_wiki[mask].mean()

ct_df = pd.DataFrame.from_dict(citation_by_type, orient="index",
                                 columns=["citation_rate"]).sort_values(
                                 "citation_rate", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ct_df["citation_rate"].plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.axhline(y_wiki.mean(), color="red", linestyle="--", label=f"Overall mean ({y_wiki.mean():.2%})")
ax.set(title="Citation-Need Rate by Predicted Epistemic Type",
       xlabel="Epistemic Type", ylabel="P(citation needed)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("citation_rate_by_type.png", dpi=150)
plt.show()

print("\nCitation-need rate by epistemic type:")
print(ct_df.to_string())
